In [1]:
# Install packages
!pip3 install datasets pandas
!pip3 install -q "transformers==4.45.2" "huggingface_hub<1.0,>=0.23.2" "datasets<4"
SEED = 7

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
!pip3 install -q "transformers==4.45.2" "huggingface_hub<1.0,>=0.23.2" "datasets<4"


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
# import opus translation model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "Helsinki-NLP/opus-mt-nl-en"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

/home/jovyan/nlp_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jovyan/nlp_env/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [4]:
from huggingface_hub import HfApi

api = HfApi()
for repo in ["Helsinki-NLP/opus-mt-nl-en",
             "yunvs/bart-base-wikilarge-simplification",
             "BramVanroy/ul2-base-dutch-simplification-mai-2023"]:
    print(repo, api.model_info(repo).sha)

from huggingface_hub import HfApi
print(HfApi().dataset_info("igornishka/dutch-municipal-sentence-simplification").sha)

Helsinki-NLP/opus-mt-nl-en 48af999f2c59b10c05ca6e008dcedc07677a9b15
yunvs/bart-base-wikilarge-simplification 050112d6e53fbde261fdc120ce19a687a9e13086
BramVanroy/ul2-base-dutch-simplification-mai-2023 4ed2e70657f452c3bed13f7c64f126220c3b6450
5d1fd646a0aeda9dba789d3b7f4284a5b1e70266


In [5]:
MODELS = {
    "mt":    ("Helsinki-NLP/opus-mt-nl-en", "48af999f2c59b10c05ca6e008dcedc07677a9b15"),
    "en_ts": ("yunvs/bart-base-wikilarge-simplification", "050112d6e53fbde261fdc120ce19a687a9e13086"),
    "nl_ts": ("BramVanroy/ul2-base-dutch-simplification-mai-2023", "4ed2e70657f452c3bed13f7c64f126220c3b6450"),
}

name, rev = MODELS["mt"]
tokenizer = AutoTokenizer.from_pretrained(name, revision=rev)
model = AutoModelForSeq2SeqLM.from_pretrained(name, revision=rev)

/home/jovyan/nlp_env/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [6]:
!pip3 list | grep torch

pytorch-lightning             2.6.0
torch                         2.5.1
torchaudio                    2.3.1+cu118
torchmetrics                  1.8.2
torchvision                   0.18.1+cu118


In [7]:
from datasets import load_dataset

ds = load_dataset(
    "igornishka/dutch-municipal-sentence-simplification",
    revision="5d1fd646a0aeda9dba789d3b7f4284a5b1e70266", 
)
df = ds["train"].to_pandas()
print(df.columns)   # ['Document ID', 'Complex', 'Simple']
print(len(df))

Index(['Document ID', 'Complex', 'Simple'], dtype='object')
1310


In [8]:
# Load data from SimPA
import pandas as pd
from pathlib import Path

def read_lines(p):
    return Path(p).read_text(encoding="utf-8").splitlines()

d = "data/simpa/"

# Syntactic: 1 reference per sentence
ss = pd.DataFrame({"src": read_lines(d + "ss.original"),
                   "ss_ref": read_lines(d + "ss.simplified")})

# Lexical: 3 references per sentence, grouped by original
ls = pd.DataFrame({"src": read_lines(d + "ls.original"),
                   "ls_ref": read_lines(d + "ls.simplified")})
ls_refs = ls.groupby("src", sort=False)["ls_ref"].apply(list)

simpa = ss.join(ls_refs, on="src")
print(len(simpa), simpa["ls_ref"].str.len().value_counts())   # expect 1100, all 3

1100 ls_ref
3.0    876
Name: count, dtype: int64


In [9]:
# Dataset statistics
def stats(src, ref, name, doc_id=None):
    s_len = src.str.split().str.len()
    r_len = ref.str.split().str.len()
    return {
        "dataset": name,
        "documents": doc_id.nunique() if doc_id is not None else "-",
        "pairs": len(src),
        "words (src)": int(s_len.sum()),
        "src tokens/sent": round(s_len.mean(), 1),
        "ref tokens/sent": round(r_len.mean(), 1),
        "compression": round((r_len / s_len).mean(), 2),
        "% unchanged": round((src.str.strip() == ref.str.strip()).mean() * 100, 1),
    }

table = pd.DataFrame([
    stats(df["Complex"], df["Simple"], "Amsterdam (nl)", doc_id=df["Document ID"]),
    stats(simpa["src"], simpa["ss_ref"], "SimPA (en, ss)"),
])
print(table.to_latex(index=False))   # paste straight into Overleaf


\begin{tabular}{llrrrrrr}
\toprule
dataset & documents & pairs & words (src) & src tokens/sent & ref tokens/sent & compression & % unchanged \\
\midrule
Amsterdam (nl) & 48 & 1310 & 21312 & 16.300000 & 15.100000 & 0.970000 & 0.800000 \\
SimPA (en, ss) & - & 1100 & 30603 & 27.800000 & 26.200000 & 0.950000 & 1.100000 \\
\bottomrule
\end{tabular}



## Human evaluation sample

In [10]:
import importlib

missing = [pkg for pkg in ("datasets", "pandas") if importlib.util.find_spec(pkg) is None]
if missing:
    !pip3 install -q {' '.join(missing)}

SEED = 42
N = 60
DATASET = "igornishka/dutch-municipal-sentence-simplification"
REVISION = "5d1fd646a0aeda9dba789d3b7f4284a5b1e70266"  # pinned commit, do not change

In [11]:
from datasets import load_dataset
import pandas as pd

raw = load_dataset(DATASET, revision=REVISION)["train"].to_pandas()
raw["row_id"] = raw.index  # original row index, so every sentence stays traceable

print(f"total pairs: {len(raw)}")
print(f"unique documents: {raw['Document ID'].nunique()}")

total pairs: 1310
unique documents: 48


In [12]:
clean = raw.copy()
clean["Complex"] = clean["Complex"].str.strip()
clean["Simple"] = clean["Simple"].str.strip()
print(f"start: {len(clean)}")

clean = clean[clean["Complex"].str.split().str.len() >= 12]
print(f"after min 12 whitespace tokens: {len(clean)}")

clean = clean[clean["Complex"].str.match(r".*[.?!]$", na=False)]
print(f"after ends in . ? !: {len(clean)}")

GAP_PATTERN = r"\s,|naar \.|\s\.$|via\.$"
clean = clean[~clean["Complex"].str.contains(GAP_PATTERN, regex=True, na=False)]
print(f"after removing gaps from stripped emails/URLs: {len(clean)}")

clean = clean[~clean["Complex"].str.contains(r"\[", regex=True, na=False)]
print(f"after removing anonymisation placeholders: {len(clean)}")

clean = clean[clean["Complex"] != clean["Simple"]]
print(f"after Complex != Simple: {len(clean)}")

clean = clean.drop_duplicates(subset="Complex")
print(f"after dropping duplicate Complex sentences: {len(clean)}")

start: 1310
after min 12 whitespace tokens: 930
after ends in . ? !: 858
after removing gaps from stripped emails/URLs: 848
after removing anonymisation placeholders: 823
after Complex != Simple: 823
after dropping duplicate Complex sentences: 814


In [13]:
CAP = 3
capped = clean.sample(frac=1, random_state=SEED).groupby("Document ID").head(CAP)

if len(capped) < N:
    CAP = 4
    print(f"fewer than {N} rows after capping at 3 per document; raising cap to {CAP}")
    capped = clean.sample(frac=1, random_state=SEED).groupby("Document ID").head(CAP)

sample = capped.sample(n=N, random_state=SEED).sort_values("row_id").reset_index(drop=True)

sample = sample.rename(columns={
    "Document ID": "doc_id",
    "Complex": "complex_nl",
    "Simple": "simple_nl",
})
sample["item_id"] = [f"s{i:02d}" for i in range(1, N + 1)]
sample = sample[["item_id", "row_id", "doc_id", "complex_nl", "simple_nl"]]

sample.to_csv("human_eval_sample.csv", index=False)
print(f"saved {len(sample)} rows to human_eval_sample.csv")

saved 60 rows to human_eval_sample.csv


In [14]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)

tokens_per_sentence = sample["complex_nl"].str.split().str.len()

print(f"documents in sample: {sample['doc_id'].nunique()}")
print("sentences per document:")
print(sample.groupby("doc_id").size())
print(f"tokens per sentence - mean: {tokens_per_sentence.mean():.1f}, "
      f"min: {tokens_per_sentence.min()}, max: {tokens_per_sentence.max()}")

sample

documents in sample: 36
sentences per document:
doc_id
1     2
4     3
5     1
6     1
7     1
9     1
11    1
13    3
14    1
16    1
17    1
18    2
20    2
21    3
22    1
24    2
25    1
26    2
28    2
29    1
30    2
31    1
32    2
33    2
34    1
35    3
36    1
38    2
39    3
41    1
42    1
43    2
44    1
46    2
47    1
48    3
dtype: int64
tokens per sentence - mean: 18.7, min: 12, max: 38


,item_id,row_id,doc_id,complex_nl,simple_nl
0,s01,3,1,Ook komt naar voren dat verschillende bewoners niet bekend zijn met huidige vervoersalternatieven zoals de Buurtbus of niet weten hoe ze hiervan gebruik kunnen maken.,Of niet weten hoe ze dat kunnen gebruiken.
1,s02,4,1,"Daarnaast geven de meeste bewoners aan buslijn 21 terug te willen in de buurt, of een betere verbinding met de dichtstbijzijnde bushalte.",Daarnaast willen de meeste bewoners buslijn 21 terug in de buurt. Of een betere verbinding met de dichtstbijzijnde bushalte.
2,s03,11,4,"ma t/m vrij: 17:00 tot 08:00 uur, za en zo en nat.",maandag tot en met vrijdag: 17:00 tot 08:00 uur
3,s04,12,4,"GGD Amsterdam: 020-55 55 202 ma t/m vrij: 09.00 - 18.00 uur, za en zo 10.00 -17.00 uur.",GGD Amsterdam: 0205 5 55 202 ma t/m vrij: 09.00 - 18.00 uur
4,s05,16,4,"Pak het vuile wasgoed (alleen in gesloten vuilniszak), de vuile vaat, afval etc.","Pak het vuile wasgoed (alleen in gesloten vuilniszak), de vuile vaat, afval enzovoort van de kamer dat bij de deur staat."
5,s06,21,5,"Door het coronavirus worden er extra maatregelen genomen, zodat u coronaproof/veilig kunt stemmen.",Door het coronavirus hebben we extra maatregelen zodat u veilig kunt stemmen.
6,s07,35,6,Dat betekent dat werkzaamheden voor Weesp zoveel mogelijk in Weesp moeten worden uitgevoerd.,Dat betekent dat werkzaamheden voor Weesp zoveel mogelijk in Weesp moeten gebeuren.
7,s08,75,7,Dan kan het zijn dat u tijdens uw arbeidsongeschiktheid reiskosten maakt in verband met uw re-integratie.,"Als u langer dan 6 weken volledig arbeidsongeschikt bent, maakt u misschien reiskosten tijdens uw arbeidsongeschiktheid vanwege uw re-integratie."
8,s09,85,9,"Uw leidinggevende heeft aangegeven dat u op grond van artikel 7.1, lid 7 PGA bij uw langdurige verzuim uw volledige salaris en salaristoelagen blijft houden.",Uw leidinggevende gaf aan dat u bij uw langdurige verzuim uw volledige salaris en salaristoelagen blijft houden.
9,s10,91,11,Met zonnepanelen op uw dak wekt u uw eigen duurzame energie op en bespaart u gemiddeld tussen de 300 en 600 euro per jaar op uw energierekening.,"Met zonnepanelen op uw dak wekt u uw eigen duurzame energie op. U bespaart daarmee gemiddeld tussen de € 300,- en € 600,- per jaar op uw energierekening."


## Simplifier evaluation (controls)

Run this section on a GPU machine (model downloads + inference over ~1300-2400 sentences on CPU would be slow). SARI is reimplemented dependency-free below, validated against `easse`'s `corpus_sari()` defaults.

In [15]:
import re
from collections import Counter
from functools import lru_cache

# SARI (Xu et al. 2016), reimplemented dependency-free to match easse's
# corpus_sari() defaults exactly (macro-averaged, F1 for deletion,
# lowercase + sacrebleu's "13a" tokenizer). Verified against easse on
# plain English, multi-reference, Dutch, and punctuation/URL cases.
NGRAM_ORDER = 4
_RE_STEPS = [
    (re.compile(r'([\{-\~\[-\` -\&\(-\+\:-\@\/])'), r' \1 '),
    (re.compile(r'([^0-9])([\.,])'), r'\1 \2 '),
    (re.compile(r'([\.,])([^0-9])'), r' \1 \2'),
    (re.compile(r'([0-9])(-)'), r'\1 \2 '),
]

@lru_cache(maxsize=2**16)
def tokenize_13a(line: str) -> str:
    line = line.replace('<skipped>', '').replace('-\n', '').replace('\n', ' ')
    if '&' in line:
        line = line.replace('&quot;', '"').replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>')
    line = f' {line} '
    for rx, repl in _RE_STEPS:
        line = rx.sub(repl, line)
    return ' '.join(line.split())

def normalize(sentence: str) -> str:
    return tokenize_13a(sentence.lower())

def extract_ngrams(line: str, max_order: int = NGRAM_ORDER):
    tokens = line.split()
    return [Counter(" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)) for n in range(1, max_order + 1)]

def multiply_counter(c, v):
    return Counter({k: cnt * v for k, cnt in c.items()})

def _prf1(correct, total, ref_total):
    p = correct / total if total > 0 else 0.0
    r = correct / ref_total if ref_total > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p > 0 and r > 0) else 0.0
    return p, r, f1

def corpus_sari(orig_sents, sys_sents, refs_sents) -> float:
    """refs_sents shape = (n_references, n_samples): ONE outer list per reference.
    For a single reference per sample use refs_sents = [list_of_refs]."""
    assert len(orig_sents) == len(sys_sents)
    assert all(len(r) == len(orig_sents) for r in refs_sents)
    orig_sents = [normalize(s) for s in orig_sents]
    sys_sents = [normalize(s) for s in sys_sents]
    refs_sents = [[normalize(s) for s in rl] for rl in refs_sents]

    add_c=[0]*4; add_t=[0]*4; add_rt=[0]*4
    keep_c=[0]*4; keep_t=[0]*4; keep_rt=[0]*4
    del_c=[0]*4; del_t=[0]*4; del_rt=[0]*4

    for orig_sent, sys_sent, *ref_sents in zip(orig_sents, sys_sents, *refs_sents):
        orig_ng = extract_ngrams(orig_sent)
        sys_ng = extract_ngrams(sys_sent)
        refs_ng = [Counter() for _ in range(4)]
        for ref_sent in ref_sents:
            rng = extract_ngrams(ref_sent)
            for n in range(4):
                refs_ng[n] += rng[n]
        num_refs = len(ref_sents)
        for n in range(4):
            sys_not_orig = set(sys_ng[n]) - set(orig_ng[n])
            add_t[n] += len(sys_not_orig)
            ref_not_orig = set(refs_ng[n]) - set(orig_ng[n])
            add_rt[n] += len(ref_not_orig)
            add_c[n] += len(sys_not_orig & set(refs_ng[n]))

            orig_and_sys = multiply_counter(orig_ng[n], num_refs) & multiply_counter(sys_ng[n], num_refs)
            keep_t[n] += sum(orig_and_sys.values())
            orig_and_ref = multiply_counter(orig_ng[n], num_refs) & refs_ng[n]
            keep_rt[n] += sum(orig_and_ref.values())
            keep_c[n] += sum((orig_and_sys & orig_and_ref).values())

            orig_not_sys = multiply_counter(orig_ng[n], num_refs) - multiply_counter(sys_ng[n], num_refs)
            del_t[n] += sum(orig_not_sys.values())
            orig_not_ref = multiply_counter(orig_ng[n], num_refs) - refs_ng[n]
            del_rt[n] += sum(orig_not_ref.values())
            del_c[n] += sum((orig_not_sys & orig_not_ref).values())

    add_f1=keep_f1=del_f1=0.0
    for n in range(4):
        add_f1 += _prf1(add_c[n], add_t[n], add_rt[n])[2] / 4
        keep_f1 += _prf1(keep_c[n], keep_t[n], keep_rt[n])[2] / 4
        del_f1 += _prf1(del_c[n], del_t[n], del_rt[n])[2] / 4
    return 100.0 * (add_f1 + keep_f1 + del_f1) / 3


In [16]:
import torch
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

_loaded = {}
def load_model(key):
    if key not in _loaded:
        name, rev = MODELS[key]
        tok = AutoTokenizer.from_pretrained(name, revision=rev)
        m = AutoModelForSeq2SeqLM.from_pretrained(name, revision=rev).to(DEVICE).eval()
        _loaded[key] = (tok, m)
    return _loaded[key]

@torch.no_grad()
def batch_generate(key, texts, batch_size=32, max_length=128, num_beams=4):
    tok, m = load_model(key)
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc=key):
        batch = list(texts[i:i+batch_size])
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(DEVICE)
        gen = m.generate(**enc, max_length=max_length, num_beams=num_beams)
        out.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return out


device: cuda


In [17]:
# Item 1: Dutch UL2 model on Amsterdam -> SARI against Dutch references
ul2_out = batch_generate("nl_ts", df["Complex"].tolist())
sari_ul2_amsterdam = corpus_sari(df["Complex"].tolist(), ul2_out, [df["Simple"].tolist()])
print("UL2 on Amsterdam SARI:", sari_ul2_amsterdam)


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/home/jovyan/nlp_env/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:558: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
nl_ts: 100%|██████████| 41/41 

UL2 on Amsterdam SARI: 40.10313060267734


In [19]:
# Item 2: English BART model on SimPA -> SARI against the 3 lexical references (main English test)
bart_out_simpa = batch_generate("en_ts", simpa["src"].tolist())
valid = simpa["ls_ref"].apply(lambda x: isinstance(x, list) and len(x) == 3)
print(f"valid rows with 3 refs: {valid.sum()} / {len(simpa)}")
src_valid = simpa.loc[valid, "src"].tolist()
bart_valid = [o for o, v in zip(bart_out_simpa, valid) if v]
refs_3 = list(map(list, zip(*simpa.loc[valid, "ls_ref"])))
sari_bart_simpa = corpus_sari(src_valid, bart_valid, refs_3)
print("BART on SimPA SARI (3 refs):", sari_bart_simpa)

en_ts: 100%|██████████| 35/35 [00:57<00:00,  1.65s/it]


valid rows with 3 refs: 876 / 1100
BART on SimPA SARI (3 refs): 34.83797184990579


In [20]:
# Item 3: English BART model on MT-translated Amsterdam -> SARI (secondary test).
# References here are machine-translated (pseudo-references), so they carry MT noise
# on top of whatever noise the simplifier itself introduces -- treat this score as
# less reliable than items 1-2, per the README caveat.
mt_complex_en = batch_generate("mt", df["Complex"].tolist())
mt_simple_en = batch_generate("mt", df["Simple"].tolist())  # pseudo-reference
bart_out_amsterdam_mt = batch_generate("en_ts", mt_complex_en)
sari_bart_amsterdam_mt = corpus_sari(mt_complex_en, bart_out_amsterdam_mt, [mt_simple_en])
print("BART on MT-translated Amsterdam SARI (pseudo-refs, MT noise caveat):", sari_bart_amsterdam_mt)


/home/jovyan/nlp_env/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
en_ts: 100%|██████████| 41/41 [00:51<00:00,  1.25s/it]


BART on MT-translated Amsterdam SARI (pseudo-refs, MT noise caveat): 35.61240552891106


## Baselines

In [ ]:
# Baseline 1: Translation only (opus-mt-nl-en, no simplification)
baseline_translation_only = mt_complex_en
print("Baseline (translation only): reused mt_complex_en, n =", len(baseline_translation_only))

# Baseline 2: Identity (copy the source, unprocessed Dutch -- a rock-bottom
# "do nothing" floor; language-mismatched against the English pseudo-refs,
# so its automatic-metric scores are expected to be very low/near-zero)
baseline_identity = df["Complex"].tolist()
print("Baseline (identity): copied df[\'Complex\'] unchanged, n =", len(baseline_identity))


## Pipelines

In [ ]:
# Pipeline 1: translate -> simplify (reuses Item 3's MT + BART outputs)
pipeline1_out = bart_out_amsterdam_mt
print("Pipeline 1 (translate->simplify): reused bart_out_amsterdam_mt, n =", len(pipeline1_out))

# Pipeline 2: simplify -> translate (UL2 simplifies Dutch, then MT translates to English)
pipeline2_out = batch_generate("mt", ul2_out)
print("Pipeline 2 (simplify->translate) sample:", pipeline2_out[0])

# Pseudo-references: Dutch Simple sentences machine-translated to English (reused from Item 3)
pseudo_refs_en = mt_simple_en
print("Pseudo-references: reused mt_simple_en, n =", len(pseudo_refs_en))


## Automatic pipeline metrics

In [ ]:
sari_pipeline1 = corpus_sari(mt_complex_en, pipeline1_out, [pseudo_refs_en])
sari_pipeline2 = corpus_sari(mt_complex_en, pipeline2_out, [pseudo_refs_en])
sari_translation_only = corpus_sari(mt_complex_en, baseline_translation_only, [pseudo_refs_en])
print("SARI - Pipeline 1 (translate->simplify):", sari_pipeline1)
print("SARI - Pipeline 2 (simplify->translate):", sari_pipeline2)
print("SARI - Translation-only baseline:", sari_translation_only)


In [ ]:
import re as _re

def count_syllables(word):
    word = _re.sub(r"[^a-z]", "", word.lower())
    if not word:
        return 0
    n = len(_re.findall(r"[aeiouy]+", word))
    if word.endswith("e") and not word.endswith("le") and n > 1:
        n -= 1
    return max(n, 1)

def fkgl(texts):
    """Flesch-Kincaid Grade Level. Syllable counting is a vowel-group heuristic
    (no dictionary lookup), so treat scores as approximate, not exact."""
    total_words = total_sentences = total_syllables = 0
    for t in texts:
        sentences = [s for s in _re.split(r"[.!?]+", t) if s.strip()]
        words = _re.findall(r"[A-Za-z\']+", t)
        total_sentences += max(len(sentences), 1)
        total_words += len(words)
        total_syllables += sum(count_syllables(w) for w in words)
    if total_words == 0:
        return float("nan")
    return 0.39 * (total_words / total_sentences) + 11.8 * (total_syllables / total_words) - 15.59

print("FKGL - Translation-only baseline:", fkgl(baseline_translation_only))
print("FKGL - Pipeline 1 (translate->simplify):", fkgl(pipeline1_out))
print("FKGL - Pipeline 2 (simplify->translate):", fkgl(pipeline2_out))
print("FKGL - Pseudo-reference (Dutch Simple, MT\'d):", fkgl(pseudo_refs_en))


In [ ]:
!pip install -q bert-score
from bert_score import score as bert_score_fn

# Meaning preservation: how much does each pipeline\'s output still say the same
# thing as the translation-only baseline (same source, no simplification)?
_, _, f1_p1 = bert_score_fn(pipeline1_out, baseline_translation_only, lang="en", verbose=False)
_, _, f1_p2 = bert_score_fn(pipeline2_out, baseline_translation_only, lang="en", verbose=False)
bertscore_pipeline1 = f1_p1.mean().item()
bertscore_pipeline2 = f1_p2.mean().item()
print("BERTScore F1 - Pipeline 1 vs translation-only:", bertscore_pipeline1)
print("BERTScore F1 - Pipeline 2 vs translation-only:", bertscore_pipeline2)


In [ ]:
import re as _re2

_NUM_RE = _re2.compile(r"\b\d[\d.,]*\b")
_DATE_RE = _re2.compile(
    r"\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b|"
    r"\b\d{1,2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{2,4}\b",
    _re2.IGNORECASE,
)
# Capitalized, non-sentence-initial word as a cheap proxy for named entities
# (no real NER here -- this is the "regex check" the README asks for, not a
# replacement for an entity tagger).
_CAP_RE = _re2.compile(r"(?<!^)(?<![.!?]\s)\b[A-Z][a-z]+\b")

def extract_entities(text):
    nums = set(_NUM_RE.findall(text))
    dates = set(_DATE_RE.findall(text))
    caps = set(_CAP_RE.findall(text))
    return nums, dates, caps

def survival_rate(orig_texts, sys_texts):
    num_total = num_kept = date_total = date_kept = cap_total = cap_kept = 0
    for orig, sys in zip(orig_texts, sys_texts):
        o_nums, o_dates, o_caps = extract_entities(orig)
        num_total += len(o_nums); num_kept += sum(1 for x in o_nums if x in sys)
        date_total += len(o_dates); date_kept += sum(1 for x in o_dates if x in sys)
        cap_total += len(o_caps); cap_kept += sum(1 for x in o_caps if x in sys)
    pct = lambda k, t: round(100 * k / t, 1) if t else float("nan")
    return {
        "numbers_pct": pct(num_kept, num_total),
        "dates_pct": pct(date_kept, date_total),
        "capitalized_words_pct(proxy for names)": pct(cap_kept, cap_total),
    }

print("Entity survival - Translation-only baseline:", survival_rate(mt_complex_en, baseline_translation_only))
print("Entity survival - Pipeline 1 (translate->simplify):", survival_rate(mt_complex_en, pipeline1_out))
print("Entity survival - Pipeline 2 (simplify->translate):", survival_rate(mt_complex_en, pipeline2_out))


In [ ]:
summary = pd.DataFrame([
    {"system": "Translation only (baseline)", "SARI": sari_translation_only, "FKGL": fkgl(baseline_translation_only), "BERTScore F1": 1.0},
    {"system": "Pipeline 1 (translate->simplify)", "SARI": sari_pipeline1, "FKGL": fkgl(pipeline1_out), "BERTScore F1": bertscore_pipeline1},
    {"system": "Pipeline 2 (simplify->translate)", "SARI": sari_pipeline2, "FKGL": fkgl(pipeline2_out), "BERTScore F1": bertscore_pipeline2},
])
print(summary.to_string(index=False))
print()
print(summary.to_latex(index=False, float_format="%.2f"))


## Human-eval website data export (pipeline outputs on the 60-sentence sample)

In [ ]:
# Run both pipelines on the 60-sentence human-eval sample (for the eval website)
import json

sample = pd.read_csv("human_eval_sample.csv")
print(len(sample), "human-eval items")

mt_sample_en = batch_generate("mt", sample["complex_nl"].tolist())
sample_p1 = batch_generate("en_ts", mt_sample_en)          # Pipeline 1: translate -> simplify
sample_ul2 = batch_generate("nl_ts", sample["complex_nl"].tolist())
sample_p2 = batch_generate("mt", sample_ul2)                # Pipeline 2: simplify -> translate

records = []
for i, row in sample.iterrows():
    records.append({
        "item_id": row["item_id"],
        "doc_id": int(row["doc_id"]),
        "complex_nl": row["complex_nl"],
        "simple_nl": row["simple_nl"],
        "pipeline1_translate_then_simplify": sample_p1[i],
        "pipeline2_simplify_then_translate": sample_p2[i],
    })

with open("human_eval_pipeline_outputs.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print("saved", len(records), "records to human_eval_pipeline_outputs.json")
